# Alternative Autoencoder Architectures Across All 10 Splits

This notebook loads the existing scaled v5 parquet datasets, trains all six autoencoder architectures using the parameters from notebook 05e, and evaluates them consistently across the same ten flight-level splits used in the LSTM-AE notebook.

**Architectures:** LSTM-AE, TCN-AE, Transformer-AE, CNN-LSTM-AE, VAE-LSTM, and BiLSTM-AE.

## 1. Imports

In [25]:
from pathlib import Path
import gc
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.12.1


## 2. Paths and device

In [26]:
# Project paths
PROJECT_DIR = Path(
    "/Users/arjun/Documents/UAV thesis/uav-thesis/"
    "uav_px4_dataset_project"
)

DATA_VERSION = "v5_voltage_random10_new_features"

INPUT_ROOT = PROJECT_DIR / f"model_ready_no_dr_{DATA_VERSION}"

EXPERIMENT_NAME = f"alternative_ae_10splits_{DATA_VERSION}"
MASTER_OUTPUT_DIR = (
    PROJECT_DIR / "model_outputs" / EXPERIMENT_NAME
)

MASTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Input root :", INPUT_ROOT)
print("Output root:", MASTER_OUTPUT_DIR)
print("Device     :", device)

if not INPUT_ROOT.exists():
    raise FileNotFoundError(
        f"Input data folder was not found:\n{INPUT_ROOT}"
    )

Input root : /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_ready_no_dr_v5_voltage_random10_new_features
Output root: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/alternative_ae_10splits_v5_voltage_random10_new_features
Device     : mps


## 3. Splits and configuration

In [27]:
# Reproducible flight-level splits
SPLIT_SEEDS = [11, 22, 33, 44, 55, 66, 77, 88, 99, 111]

ALL_SPLITS = [
    f"split_{str(i + 1).zfill(2)}_seed_{seed}"
    for i, seed in enumerate(SPLIT_SEEDS)
]

SPLIT_SEED_MAP = dict(zip(ALL_SPLITS, SPLIT_SEEDS))

TARGET_COL = "target_energy_next_60s_wh"

FEATURE_COLS = [
    "battery_status__cell_count",
    "battery_status__current_a",
    "battery_status__current_filtered_a",
    "battery_status__discharged_mah",
    "battery_status__remaining",
    "battery_status__scale",
    "battery_status__voltage_filtered_v",
    "battery_status__voltage_v",
    "cumulative_energy_wh",
    "power_w",
    "time_s",
    "is_idle",
    "power_w_rolling_std_3s",
    "voltage_per_cell",
    "voltage_filtered_per_cell",
    "voltage_per_cell_suspicious",
]

# Sequence and training settings from notebook 05e
SEQ_LEN = 100
STRIDE_TRAIN = 20
STRIDE_SCORE = 10
BATCH_SIZE = 64
EPOCHS = 200
PATIENCE = 15
LEARNING_RATE = 0.001
MAX_GRAD_NORM = 1.0
MIN_DELTA = 1e-5
THRESHOLD_PERCENTILE = 95

# All six architectures used in notebook 05e
ARCHITECTURES_TO_RUN = [
    "LSTM-AE",
    "TCN-AE",
    "Transformer-AE",
    "CNN-LSTM-AE",
    "VAE-LSTM",
    "BiLSTM-AE",
]

# Useful controls for restarting or testing the notebook
SPLITS_TO_RUN = ALL_SPLITS
FORCE_RERUN = False

# Full row-level files are large. Add architecture names here only
# when their row-level anomaly scores are needed downstream.
SAVE_ROW_LEVEL_SCORES_FOR = {"CNN-LSTM-AE"}

assert len(FEATURE_COLS) == 16
assert "energy_increment_wh" not in FEATURE_COLS

print("\nSplits:", len(SPLITS_TO_RUN))
print("Architectures:", ARCHITECTURES_TO_RUN)
print("Features:", len(FEATURE_COLS))
print(
    f"Sequence length: {SEQ_LEN} rows "
    f"= {SEQ_LEN / 10:.0f} seconds at 10 Hz"
)


Splits: 10
Architectures: ['LSTM-AE', 'TCN-AE', 'Transformer-AE', 'CNN-LSTM-AE', 'VAE-LSTM', 'BiLSTM-AE']
Features: 16
Sequence length: 100 rows = 10 seconds at 10 Hz


## 4. Data helpers

In [28]:
def set_seed(seed: int) -> None:
    """Set Python, NumPy, and PyTorch seeds."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


def clear_device_cache() -> None:
    """Release Python and accelerator memory between models."""
    gc.collect()

    if device.type == "mps":
        torch.mps.empty_cache()
    elif device.type == "cuda":
        torch.cuda.empty_cache()


def safe_model_name(model_name: str) -> str:
    """Convert a display name into a safe folder name."""
    return (
        model_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


def load_split_files(
    input_dir: Path,
    split_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the scaled train, validation, and test files for one split."""
    paths = {
        "train": (
            input_dir
            / f"train_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet"
        ),
        "val": (
            input_dir
            / f"val_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet"
        ),
        "test": (
            input_dir
            / f"test_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet"
        ),
    }

    for data_split, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(
                f"Missing {data_split} file for {split_name}:\n{path}"
            )

    train_df = pd.read_parquet(paths["train"])
    val_df = pd.read_parquet(paths["val"])
    test_df = pd.read_parquet(paths["test"])

    print("Loaded:")
    print(f"  Train: {paths['train'].name} {train_df.shape}")
    print(f"  Val  : {paths['val'].name} {val_df.shape}")
    print(f"  Test : {paths['test'].name} {test_df.shape}")

    return train_df, val_df, test_df


def validate_data(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    split_name: str,
) -> None:
    """Check required columns and missing feature values."""
    required_cols = [
        "log_id",
        "dataset_label",
        TARGET_COL,
        *FEATURE_COLS,
    ]

    for data_split, df in [
        ("Train", train_df),
        ("Validation", val_df),
        ("Test", test_df),
    ]:
        missing_cols = [
            col for col in required_cols
            if col not in df.columns
        ]

        if missing_cols:
            raise ValueError(
                f"{split_name} | {data_split}: "
                f"missing columns {missing_cols}"
            )

        nan_count = int(
            df[FEATURE_COLS].isna().sum().sum()
        )

        if nan_count > 0:
            bad_cols = (
                df[FEATURE_COLS]
                .columns[df[FEATURE_COLS].isna().any()]
                .tolist()
            )
            raise ValueError(
                f"{split_name} | {data_split}: "
                f"{nan_count} NaN values in {bad_cols}"
            )

    print("Data validation passed.")


def prepare_sorted_df(df: pd.DataFrame) -> pd.DataFrame:
    """Sort rows by flight and time while preserving original order."""
    out = df.copy()
    out["_original_order"] = np.arange(len(out))
    out["log_id"] = out["log_id"].astype(str)

    return (
        out
        .sort_values(["log_id", "time_s"])
        .reset_index(drop=True)
    )


def build_window_starts(
    df_sorted: pd.DataFrame,
    seq_len: int,
    stride: int,
    normal_only: bool = False,
    clean_only: bool = False,
) -> np.ndarray:
    """
    Create sliding-window start positions without crossing flights.

    When normal_only=True, only windows containing entirely normal
    rows are retained.

    When clean_only=True, windows containing suspicious per-cell
    voltage rows are removed.
    """
    window_starts = []

    for _, group in df_sorted.groupby("log_id", sort=False):
        indices = group.index.to_numpy()
        n_rows = len(group)

        if n_rows < seq_len:
            continue

        for local_start in range(
            0,
            n_rows - seq_len + 1,
            stride,
        ):
            window_indices = indices[
                local_start : local_start + seq_len
            ]

            window_df = df_sorted.loc[window_indices]

            # Keep only fully normal windows
            if normal_only:
                normal_window = (
                    window_df["dataset_label"] == "normal"
                ).all()

                if not normal_window:
                    continue

            # Remove windows containing suspicious voltage rows
            if clean_only:
                clean_window = (
                    window_df[
                        "voltage_per_cell_suspicious"
                    ] == 0
                ).all()

                if not clean_window:
                    continue

            window_starts.append(window_indices[0])

    return np.asarray(
        window_starts,
        dtype=np.int64,
    )

def restore_original_order(
    scored_df: pd.DataFrame,
) -> pd.DataFrame:
    """Return scored rows in the same order as the source parquet."""
    return (
        scored_df
        .sort_values("_original_order")
        .drop(columns=["_original_order"])
        .reset_index(drop=True)
    )

## 5. Dataset classes

In [29]:
class UAVTrainSequenceDataset(Dataset):
    """Return fixed-length feature sequences for model training."""

    def __init__(
        self,
        data_array: np.ndarray,
        window_starts: np.ndarray,
        seq_len: int,
    ):
        self.data_array = data_array
        self.window_starts = window_starts
        self.seq_len = seq_len

    def __len__(self) -> int:
        return len(self.window_starts)

    def __getitem__(self, idx: int) -> torch.Tensor:
        start = self.window_starts[idx]

        return torch.tensor(
            self.data_array[start : start + self.seq_len],
            dtype=torch.float32,
        )


class UAVScoringSequenceDataset(Dataset):
    """Return feature sequences together with their row start index."""

    def __init__(
        self,
        data_array: np.ndarray,
        window_starts: np.ndarray,
        seq_len: int,
    ):
        self.data_array = data_array
        self.window_starts = window_starts
        self.seq_len = seq_len

    def __len__(self) -> int:
        return len(self.window_starts)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        start = self.window_starts[idx]

        sequence = torch.tensor(
            self.data_array[start : start + self.seq_len],
            dtype=torch.float32,
        )

        return sequence, start

## 6. Autoencoder architectures

In [30]:
class LSTMAutoencoder(nn.Module):
    """LSTM encoder-decoder with a compact latent bottleneck."""

    def __init__(
        self,
        input_size: int,
        hidden_size: int = 64,
        latent_size: int = 32,
    ):
        super().__init__()

        self.encoder_lstm = nn.LSTM(
            input_size,
            hidden_size,
            batch_first=True,
        )
        self.encoder_fc = nn.Linear(
            hidden_size,
            latent_size,
        )

        self.decoder_input_fc = nn.Linear(
            latent_size,
            hidden_size,
        )
        self.decoder_h0_fc = nn.Linear(
            latent_size,
            hidden_size,
        )
        self.decoder_c0_fc = nn.Linear(
            latent_size,
            hidden_size,
        )

        self.decoder_lstm = nn.LSTM(
            hidden_size,
            hidden_size,
            batch_first=True,
        )
        self.output_layer = nn.Linear(
            hidden_size,
            input_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (hidden, _) = self.encoder_lstm(x)
        latent = self.encoder_fc(hidden[-1])

        h0 = self.decoder_h0_fc(latent).unsqueeze(0)
        c0 = self.decoder_c0_fc(latent).unsqueeze(0)

        decoder_input = (
            self.decoder_input_fc(latent)
            .unsqueeze(1)
            .repeat(1, x.size(1), 1)
        )

        decoder_output, _ = self.decoder_lstm(
            decoder_input,
            (h0, c0),
        )

        return self.output_layer(decoder_output)


class Chomp1d(nn.Module):
    """Remove right-side padding from a causal convolution."""

    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.chomp_size == 0:
            return x

        return x[:, :, :-self.chomp_size].contiguous()


class TCNBlock(nn.Module):
    """Residual temporal convolution block."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        dilation: int,
        dropout: float = 0.1,
    ):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv = nn.Sequential(
            nn.utils.weight_norm(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size,
                    dilation=dilation,
                    padding=padding,
                )
            ),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.downsample = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else None
        )

        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.conv(x)

        residual = (
            x
            if self.downsample is None
            else self.downsample(x)
        )

        return self.relu(out + residual)


class TCNAutoencoder(nn.Module):
    """Temporal convolutional autoencoder."""

    def __init__(
        self,
        input_size: int,
        channels: tuple[int, ...] = (64, 32, 16),
        kernel_size: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()

        encoder_layers = []
        in_channels = input_size

        for level, out_channels in enumerate(channels):
            encoder_layers.append(
                TCNBlock(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    dilation=2 ** level,
                    dropout=dropout,
                )
            )
            in_channels = out_channels

        self.encoder = nn.Sequential(*encoder_layers)

        decoder_channels = (
            tuple(reversed(channels))
            + (input_size,)
        )

        decoder_layers = []
        in_channels = channels[-1]

        for level, out_channels in enumerate(decoder_channels):
            dilation = max(
                1,
                2 ** (len(channels) - level - 1),
            )

            decoder_layers.append(
                TCNBlock(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_channels = out_channels

        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        x = self.encoder(x)
        x = self.decoder(x)

        return x.transpose(1, 2)


class TransformerAutoencoder(nn.Module):
    """Transformer encoder used to reconstruct the full sequence."""

    def __init__(
        self,
        input_size: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dropout: float = 0.1,
        max_seq_len: int = 200,
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            input_size,
            d_model,
        )

        self.position_encoding = nn.Parameter(
            torch.randn(1, max_seq_len, d_model) * 0.1
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.output_projection = nn.Linear(
            d_model,
            input_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)

        if seq_len > self.position_encoding.size(1):
            raise ValueError(
                f"Sequence length {seq_len} exceeds "
                f"maximum {self.position_encoding.size(1)}"
            )

        projected = (
            self.input_projection(x)
            + self.position_encoding[:, :seq_len, :]
        )

        encoded = self.transformer(projected)

        return self.output_projection(encoded)


class CNNLSTMAutoencoder(nn.Module):
    """CNN feature extractor followed by an LSTM autoencoder."""

    def __init__(
        self,
        input_size: int,
        cnn_channels: int = 32,
        lstm_hidden: int = 64,
        latent_size: int = 32,
    ):
        super().__init__()

        self.cnn_encoder = nn.Sequential(
            nn.Conv1d(
                input_size,
                cnn_channels,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.Conv1d(
                cnn_channels,
                cnn_channels,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
        )

        self.lstm_encoder = nn.LSTM(
            cnn_channels,
            lstm_hidden,
            batch_first=True,
        )
        self.latent_fc = nn.Linear(
            lstm_hidden,
            latent_size,
        )
        self.decoder_fc = nn.Linear(
            latent_size,
            lstm_hidden,
        )
        self.lstm_decoder = nn.LSTM(
            lstm_hidden,
            lstm_hidden,
            batch_first=True,
        )
        self.output_layer = nn.Linear(
            lstm_hidden,
            input_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        cnn_output = (
            self.cnn_encoder(x.transpose(1, 2))
            .transpose(1, 2)
        )

        _, (hidden, _) = self.lstm_encoder(cnn_output)
        latent = self.latent_fc(hidden[-1])

        decoder_input = (
            self.decoder_fc(latent)
            .unsqueeze(1)
            .repeat(1, x.size(1), 1)
        )

        decoder_output, _ = self.lstm_decoder(
            decoder_input
        )

        return self.output_layer(decoder_output)


class VAELSTMAutoencoder(nn.Module):
    """Variational LSTM autoencoder with a small KL penalty."""

    def __init__(
        self,
        input_size: int,
        hidden_size: int = 64,
        latent_size: int = 32,
        beta: float = 0.001,
    ):
        super().__init__()

        self.beta = beta

        self.encoder_lstm = nn.LSTM(
            input_size,
            hidden_size,
            batch_first=True,
        )
        self.mu_fc = nn.Linear(
            hidden_size,
            latent_size,
        )
        self.logvar_fc = nn.Linear(
            hidden_size,
            latent_size,
        )
        self.decoder_fc = nn.Linear(
            latent_size,
            hidden_size,
        )
        self.decoder_lstm = nn.LSTM(
            hidden_size,
            hidden_size,
            batch_first=True,
        )
        self.output_layer = nn.Linear(
            hidden_size,
            input_size,
        )

    @staticmethod
    def reparameterise(
        mu: torch.Tensor,
        logvar: torch.Tensor,
    ) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        noise = torch.randn_like(std)

        return mu + noise * std

    def forward(
        self,
        x: torch.Tensor,
        sample: bool = True,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        _, (hidden, _) = self.encoder_lstm(x)

        mu = self.mu_fc(hidden[-1])
        logvar = self.logvar_fc(hidden[-1])

        latent = (
            self.reparameterise(mu, logvar)
            if sample
            else mu
        )

        decoder_input = (
            self.decoder_fc(latent)
            .unsqueeze(1)
            .repeat(1, x.size(1), 1)
        )

        decoder_output, _ = self.decoder_lstm(
            decoder_input
        )

        reconstruction = self.output_layer(
            decoder_output
        )

        return reconstruction, mu, logvar

    def loss(
        self,
        x: torch.Tensor,
        reconstruction: torch.Tensor,
        mu: torch.Tensor,
        logvar: torch.Tensor,
    ) -> torch.Tensor:
        reconstruction_loss = nn.functional.mse_loss(
            reconstruction,
            x,
        )

        kl_loss = -0.5 * torch.mean(
            1
            + logvar
            - mu.pow(2)
            - logvar.exp()
        )

        return reconstruction_loss + self.beta * kl_loss


class BiLSTMAutoencoder(nn.Module):
    """Bidirectional LSTM encoder with a unidirectional decoder."""

    def __init__(
        self,
        input_size: int,
        hidden_size: int = 64,
        latent_size: int = 32,
    ):
        super().__init__()

        self.encoder_lstm = nn.LSTM(
            input_size,
            hidden_size,
            batch_first=True,
            bidirectional=True,
        )
        self.encoder_fc = nn.Linear(
            hidden_size * 2,
            latent_size,
        )
        self.decoder_fc = nn.Linear(
            latent_size,
            hidden_size,
        )
        self.decoder_lstm = nn.LSTM(
            hidden_size,
            hidden_size,
            batch_first=True,
        )
        self.output_layer = nn.Linear(
            hidden_size,
            input_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (hidden, _) = self.encoder_lstm(x)

        bidirectional_hidden = torch.cat(
            [hidden[-2], hidden[-1]],
            dim=1,
        )

        latent = self.encoder_fc(
            bidirectional_hidden
        )

        decoder_input = (
            self.decoder_fc(latent)
            .unsqueeze(1)
            .repeat(1, x.size(1), 1)
        )

        decoder_output, _ = self.decoder_lstm(
            decoder_input
        )

        return self.output_layer(decoder_output)

## 7. Model helpers

In [31]:
def create_model(
    architecture: str,
    input_size: int,
) -> nn.Module:
    """Create a fresh model using the parameters from notebook 05e."""
    if architecture == "LSTM-AE":
        model = LSTMAutoencoder(
            input_size=input_size,
            hidden_size=64,
            latent_size=32,
        )

    elif architecture == "TCN-AE":
        model = TCNAutoencoder(
            input_size=input_size,
            channels=(64, 32, 16),
            kernel_size=3,
            dropout=0.1,
        )

    elif architecture == "Transformer-AE":
        model = TransformerAutoencoder(
            input_size=input_size,
            d_model=64,
            nhead=4,
            num_layers=2,
            dropout=0.1,
            max_seq_len=200,
        )

    elif architecture == "CNN-LSTM-AE":
        model = CNNLSTMAutoencoder(
            input_size=input_size,
            cnn_channels=32,
            lstm_hidden=64,
            latent_size=32,
        )

    elif architecture == "VAE-LSTM":
        model = VAELSTMAutoencoder(
            input_size=input_size,
            hidden_size=64,
            latent_size=32,
            beta=0.001,
        )

    elif architecture == "BiLSTM-AE":
        model = BiLSTMAutoencoder(
            input_size=input_size,
            hidden_size=64,
            latent_size=32,
        )

    else:
        raise ValueError(
            f"Unknown architecture: {architecture}"
        )

    return model.to(device)


def reconstruct_batch(
    model: nn.Module,
    batch_x: torch.Tensor,
    deterministic: bool = True,
) -> torch.Tensor:
    """Return only the reconstructed sequence for any architecture."""
    if isinstance(model, VAELSTMAutoencoder):
        reconstruction, _, _ = model(
            batch_x,
            sample=not deterministic,
        )
        return reconstruction

    return model(batch_x)


def calculate_batch_loss(
    model: nn.Module,
    batch_x: torch.Tensor,
    deterministic: bool,
) -> torch.Tensor:
    """Calculate the correct training or validation loss."""
    if isinstance(model, VAELSTMAutoencoder):
        reconstruction, mu, logvar = model(
            batch_x,
            sample=not deterministic,
        )
        return model.loss(
            batch_x,
            reconstruction,
            mu,
            logvar,
        )

    reconstruction = model(batch_x)

    return nn.functional.mse_loss(
        reconstruction,
        batch_x,
    )


def count_trainable_parameters(
    model: nn.Module,
) -> int:
    """Count trainable model parameters."""
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


print("Architecture parameter counts:")

for architecture in ARCHITECTURES_TO_RUN:
    check_model = create_model(
        architecture,
        input_size=len(FEATURE_COLS),
    )

    print(
        f"  {architecture:<20}: "
        f"{count_trainable_parameters(check_model):>8,}"
    )

    del check_model
    clear_device_cache()

Architecture parameter counts:
  LSTM-AE             :   63,728
  TCN-AE              :   30,144


/opt/anaconda3/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Transformer-AE      :  114,896
  CNN-LSTM-AE         :   68,272
  VAE-LSTM            :   61,584
  BiLSTM-AE           :   82,544


## 8. Training function

In [32]:
def train_autoencoder(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    model_path: Path,
) -> tuple[nn.Module, pd.DataFrame, float]:
    """Train one architecture with early stopping."""
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
    )

    best_val_loss = np.inf
    patience_counter = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_batch_losses = []

        for batch_x in train_loader:
            batch_x = batch_x.to(device)

            optimizer.zero_grad()

            loss = calculate_batch_loss(
                model,
                batch_x,
                deterministic=False,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=MAX_GRAD_NORM,
            )

            optimizer.step()
            train_batch_losses.append(loss.item())

        train_loss = float(
            np.mean(train_batch_losses)
        )

        model.eval()
        val_batch_losses = []

        with torch.no_grad():
            for batch_x in val_loader:
                batch_x = batch_x.to(device)

                val_loss = calculate_batch_loss(
                    model,
                    batch_x,
                    deterministic=True,
                )

                val_batch_losses.append(
                    val_loss.item()
                )

        val_loss = float(
            np.mean(val_batch_losses)
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "gap": val_loss - train_loss,
        })

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Train: {train_loss:.6f} | "
                f"Val: {val_loss:.6f} | "
                f"Gap: {val_loss - train_loss:+.6f}"
            )

        improved = (
            val_loss
            < best_val_loss - MIN_DELTA
        )

        if improved:
            best_val_loss = val_loss
            patience_counter = 0

            torch.save(
                model.state_dict(),
                model_path,
            )
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(
                f"Early stopping at epoch {epoch}. "
                f"Best validation loss: "
                f"{best_val_loss:.6f}"
            )
            break

    if not model_path.exists():
        raise FileNotFoundError(
            f"Best model was not saved:\n{model_path}"
        )

    model.load_state_dict(
        torch.load(
            model_path,
            map_location=device,
            weights_only=True,
        )
    )

    model.to(device)
    model.eval()

    history_df = pd.DataFrame(history)

    return model, history_df, best_val_loss

## 9. Scoring and metrics

In [33]:
def score_dataframe(
    df_sorted: pd.DataFrame,
    model: nn.Module,
) -> pd.DataFrame:
    """Assign an averaged per-row reconstruction error."""
    feature_array = df_sorted[
        FEATURE_COLS
    ].to_numpy(dtype=np.float32)

    window_starts = build_window_starts(
        df_sorted,
        seq_len=SEQ_LEN,
        stride=STRIDE_SCORE,
        normal_only=False,
        clean_only=False,
    )

    if len(window_starts) == 0:
        raise ValueError(
            "No scoring windows were created."
        )

    dataset = UAVScoringSequenceDataset(
        feature_array,
        window_starts,
        SEQ_LEN,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )

    score_sum = np.zeros(
        len(df_sorted),
        dtype=np.float64,
    )
    score_count = np.zeros(
        len(df_sorted),
        dtype=np.float64,
    )

    model.eval()

    with torch.no_grad():
        for batch_x, batch_starts in loader:
            batch_x = batch_x.to(device)

            reconstruction = reconstruct_batch(
                model,
                batch_x,
                deterministic=True,
            )

            step_errors = (
                (batch_x - reconstruction)
                .pow(2)
                .mean(dim=2)
                .cpu()
                .numpy()
            )

            for row, start in enumerate(
                batch_starts.numpy()
            ):
                end = start + SEQ_LEN

                score_sum[start:end] += (
                    step_errors[row]
                )
                score_count[start:end] += 1

    anomaly_score = np.divide(
        score_sum,
        score_count,
        out=np.full(
            len(df_sorted),
            np.nan,
            dtype=np.float64,
        ),
        where=score_count > 0,
    )

    scored_df = df_sorted.copy()
    scored_df["anomaly_score"] = anomaly_score

    if scored_df["anomaly_score"].isna().any():
        scored_df["anomaly_score"] = (
            scored_df
            .groupby("log_id")["anomaly_score"]
            .transform(
                lambda values: values.ffill().bfill()
            )
        )

    if scored_df["anomaly_score"].isna().any():
        normal_median = scored_df.loc[
            scored_df["dataset_label"] == "normal",
            "anomaly_score",
        ].median()

        fallback = (
            normal_median
            if pd.notna(normal_median)
            else scored_df["anomaly_score"].median()
        )

        scored_df["anomaly_score"] = (
            scored_df["anomaly_score"]
            .fillna(fallback)
        )

    return scored_df


def create_flight_summary(
    scored_df: pd.DataFrame,
    split_name: str,
    data_split: str,
    architecture: str,
    threshold: float,
) -> pd.DataFrame:
    """Aggregate row-level scores to one row per flight."""
    flight_summary = (
        scored_df
        .groupby(
            ["log_id", "dataset_label"],
            as_index=False,
        )
        .agg(
            mean_anomaly_score=(
                "anomaly_score",
                "mean",
            ),
            max_anomaly_score=(
                "anomaly_score",
                "max",
            ),
            std_anomaly_score=(
                "anomaly_score",
                "std",
            ),
            flagged_percentage=(
                "anomaly_flag",
                "mean",
            ),
            rows=(
                "anomaly_score",
                "size",
            ),
        )
    )

    flight_summary["flagged_percentage"] *= 100
    flight_summary["split"] = split_name
    flight_summary["data_split"] = data_split
    flight_summary["architecture"] = architecture
    flight_summary["threshold"] = threshold

    return flight_summary


def compute_flight_metrics(
    flight_summary: pd.DataFrame,
    best_val_loss: float,
    n_params: int,
    training_seconds: float,
) -> dict:
    """Calculate flight-level AUROC, AUPRC, and score separation."""
    labels = (
        flight_summary["dataset_label"]
        .eq("abnormal")
        .astype(int)
    )

    scores = flight_summary[
        "mean_anomaly_score"
    ]

    normal_scores = scores[labels == 0]
    abnormal_scores = scores[labels == 1]

    auroc = np.nan
    auprc = np.nan

    if labels.nunique() == 2:
        auroc = roc_auc_score(
            labels,
            scores,
        )
        auprc = average_precision_score(
            labels,
            scores,
        )

    normal_mean = normal_scores.mean()
    abnormal_mean = abnormal_scores.mean()

    separation_ratio = (
        abnormal_mean / normal_mean
        if pd.notna(normal_mean) and normal_mean > 0
        else np.nan
    )

    return {
        "split": flight_summary["split"].iloc[0],
        "data_split": flight_summary["data_split"].iloc[0],
        "architecture": flight_summary["architecture"].iloc[0],
        "threshold": flight_summary["threshold"].iloc[0],
        "best_val_loss": best_val_loss,
        "n_params": n_params,
        "training_seconds": training_seconds,
        "n_normal_flights": int((labels == 0).sum()),
        "n_abnormal_flights": int((labels == 1).sum()),
        "mean_score_normal": normal_mean,
        "mean_score_abnormal": abnormal_mean,
        "separation_ratio": separation_ratio,
        "flight_auroc": auroc,
        "flight_auprc": auprc,
        "mean_flagged_pct_normal": (
            flight_summary.loc[
                labels == 0,
                "flagged_percentage",
            ].mean()
        ),
        "mean_flagged_pct_abnormal": (
            flight_summary.loc[
                labels == 1,
                "flagged_percentage",
            ].mean()
        ),
    }

## 10. Plot helpers

In [34]:
def save_training_curve(
    history_df: pd.DataFrame,
    plot_path: Path,
    architecture: str,
    split_name: str,
) -> None:
    """Save the train and validation loss curves."""
    plt.figure(figsize=(8, 5))

    plt.plot(
        history_df["epoch"],
        history_df["train_loss"],
        label="Train loss",
    )
    plt.plot(
        history_df["epoch"],
        history_df["val_loss"],
        label="Validation loss",
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(
        f"{architecture} Training Curve\n"
        f"{split_name}"
    )
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        plot_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()


def save_score_distribution(
    test_scored: pd.DataFrame,
    threshold: float,
    plot_path: Path,
    architecture: str,
    split_name: str,
) -> None:
    """Save normal and abnormal test-score distributions."""
    plt.figure(figsize=(8, 5))

    for label in ["normal", "abnormal"]:
        subset = test_scored[
            test_scored["dataset_label"] == label
        ]

        if len(subset) > 0:
            plt.hist(
                subset["anomaly_score"],
                bins=60,
                alpha=0.55,
                density=True,
                label=f"{label} (n={len(subset):,})",
            )

    plt.axvline(
        threshold,
        linestyle="--",
        linewidth=1.3,
        label=f"Threshold = {threshold:.4f}",
    )

    plt.xlabel("Anomaly score")
    plt.ylabel("Density")
    plt.title(
        f"{architecture} Test Score Distribution\n"
        f"{split_name}"
    )
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        plot_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()

## 11. Main 10-split training loop

In [35]:
ALL_METRICS = []
ALL_FLIGHT_SUMMARIES = []

for split_name in SPLITS_TO_RUN:
    split_seed = SPLIT_SEED_MAP[split_name]

    print("\n" + "=" * 100)
    print(f"STARTING SPLIT: {split_name}")
    print("=" * 100)

    input_dir = INPUT_ROOT / split_name

    train_df, val_df, test_df = load_split_files(
        input_dir,
        split_name,
    )

    validate_data(
        train_df,
        val_df,
        test_df,
        split_name,
    )

    train_sorted = prepare_sorted_df(train_df)
    val_sorted = prepare_sorted_df(val_df)
    test_sorted = prepare_sorted_df(test_df)

    train_array = train_sorted[
        FEATURE_COLS
    ].to_numpy(dtype=np.float32)

    val_array = val_sorted[
        FEATURE_COLS
    ].to_numpy(dtype=np.float32)

    # Training and validation use only clean normal windows.
    train_starts = build_window_starts(
        train_sorted,
        seq_len=SEQ_LEN,
        stride=STRIDE_TRAIN,
        normal_only=True
    )

    val_starts = build_window_starts(
        val_sorted,
        seq_len=SEQ_LEN,
        stride=STRIDE_TRAIN,
        normal_only=True
    )

    print(
        f"Clean normal training windows   : "
        f"{len(train_starts):,}"
    )
    print(
        f"Clean normal validation windows : "
        f"{len(val_starts):,}"
    )

    if len(train_starts) == 0:
        print("No training windows. Split skipped.")
        continue

    if len(val_starts) == 0:
        print("No validation windows. Split skipped.")
        continue

    train_dataset = UAVTrainSequenceDataset(
        train_array,
        train_starts,
        SEQ_LEN,
    )

    val_dataset = UAVTrainSequenceDataset(
        val_array,
        val_starts,
        SEQ_LEN,
    )

    for architecture_index, architecture in enumerate(
        ARCHITECTURES_TO_RUN
    ):
        print("\n" + "-" * 80)
        print(
            f"{split_name} | "
            f"{architecture_index + 1}/"
            f"{len(ARCHITECTURES_TO_RUN)} | "
            f"{architecture}"
        )
        print("-" * 80)

        architecture_dir = (
            MASTER_OUTPUT_DIR
            / split_name
            / safe_model_name(architecture)
        )
        plot_dir = architecture_dir / "plots"

        architecture_dir.mkdir(
            parents=True,
            exist_ok=True,
        )
        plot_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        model_path = (
            architecture_dir
            / "best_model.pt"
        )
        history_path = (
            architecture_dir
            / "training_history.csv"
        )
        metrics_path = (
            architecture_dir
            / "metrics.csv"
        )

        flight_paths = {
            "Train": (
                architecture_dir
                / "flight_summary_train.csv"
            ),
            "Validation": (
                architecture_dir
                / "flight_summary_validation.csv"
            ),
            "Test": (
                architecture_dir
                / "flight_summary_test.csv"
            ),
        }

        completed = (
            metrics_path.exists()
            and all(
                path.exists()
                for path in flight_paths.values()
            )
        )

        if completed and not FORCE_RERUN:
            print(
                "Completed outputs found. "
                "Loading saved results."
            )

            metrics_df = pd.read_csv(
                metrics_path
            )

            ALL_METRICS.append(metrics_df)

            for path in flight_paths.values():
                ALL_FLIGHT_SUMMARIES.append(
                    pd.read_csv(path)
                )

            continue

        # Use a reproducible but architecture-specific seed.
        model_seed = (
            split_seed * 100
            + architecture_index
        )
        set_seed(model_seed)

        loader_generator = torch.Generator()
        loader_generator.manual_seed(model_seed)

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            drop_last=False,
            num_workers=0,
            generator=loader_generator,
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            drop_last=False,
            num_workers=0,
        )

        model = create_model(
            architecture,
            input_size=len(FEATURE_COLS),
        )

        n_params = count_trainable_parameters(
            model
        )

        print(model)
        print(
            f"Trainable parameters: {n_params:,}"
        )

        start_time = time.perf_counter()

        model, history_df, best_val_loss = (
            train_autoencoder(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                model_path=model_path,
            )
        )

        training_seconds = (
            time.perf_counter() - start_time
        )

        history_df.to_csv(
            history_path,
            index=False,
        )

        save_training_curve(
            history_df=history_df,
            plot_path=(
                plot_dir
                / "training_curve.png"
            ),
            architecture=architecture,
            split_name=split_name,
        )

        print("\nScoring training data...")
        train_scored = score_dataframe(
            train_sorted,
            model,
        )

        normal_train_scores = train_scored.loc[
            train_scored["dataset_label"] == "normal",
            "anomaly_score",
        ].dropna()

        if len(normal_train_scores) == 0:
            raise ValueError(
                "No normal training scores were available "
                "for threshold calculation."
            )

        threshold = float(
            np.percentile(
                normal_train_scores,
                THRESHOLD_PERCENTILE,
            )
        )

        print(
            f"Anomaly threshold: {threshold:.6f} "
            f"({THRESHOLD_PERCENTILE}th percentile)"
        )

        print("Scoring validation data...")
        val_scored = score_dataframe(
            val_sorted,
            model,
        )

        print("Scoring test data...")
        test_scored = score_dataframe(
            test_sorted,
            model,
        )

        for scored_df in [
            train_scored,
            val_scored,
            test_scored,
        ]:
            scored_df["anomaly_flag"] = (
                scored_df["anomaly_score"]
                > threshold
            ).astype(int)

        flight_summaries = {}

        for data_split, scored_df in [
            ("Train", train_scored),
            ("Validation", val_scored),
            ("Test", test_scored),
        ]:
            flight_summary = create_flight_summary(
                scored_df=scored_df,
                split_name=split_name,
                data_split=data_split,
                architecture=architecture,
                threshold=threshold,
            )

            flight_summary.to_csv(
                flight_paths[data_split],
                index=False,
            )

            flight_summaries[data_split] = (
                flight_summary
            )
            ALL_FLIGHT_SUMMARIES.append(
                flight_summary
            )

        model_metrics = []

        for data_split in [
            "Train",
            "Validation",
            "Test",
        ]:
            metrics = compute_flight_metrics(
                flight_summary=(
                    flight_summaries[data_split]
                ),
                best_val_loss=best_val_loss,
                n_params=n_params,
                training_seconds=training_seconds,
            )

            model_metrics.append(metrics)

        metrics_df = pd.DataFrame(
            model_metrics
        )

        metrics_df.to_csv(
            metrics_path,
            index=False,
        )

        ALL_METRICS.append(metrics_df)

        save_score_distribution(
            test_scored=test_scored,
            threshold=threshold,
            plot_path=(
                plot_dir
                / "test_score_distribution.png"
            ),
            architecture=architecture,
            split_name=split_name,
        )

        if architecture in SAVE_ROW_LEVEL_SCORES_FOR:
            for data_split, scored_df in [
                ("train", train_scored),
                ("val", val_scored),
                ("test", test_scored),
            ]:
                output_df = restore_original_order(
                    scored_df
                )

                output_path = (
                    architecture_dir
                    / (
                        f"{data_split}_with_"
                        f"{safe_model_name(architecture)}_"
                        f"anomaly_score.csv"
                    )
                )

                output_df.to_csv(
                    output_path,
                    index=False,
                )

                print(
                    f"Saved row scores: {output_path}"
                )

        test_metrics = metrics_df[
            metrics_df["data_split"] == "Test"
        ].iloc[0]

        print(
            f"\nFinished {architecture} | "
            f"Test AUROC="
            f"{test_metrics['flight_auroc']:.4f} | "
            f"Test AUPRC="
            f"{test_metrics['flight_auprc']:.4f} | "
            f"Separation="
            f"{test_metrics['separation_ratio']:.2f}x"
        )

        del model
        del train_loader, val_loader
        del train_scored, val_scored, test_scored

        clear_device_cache()

    del train_df, val_df, test_df
    del train_sorted, val_sorted, test_sorted
    del train_array, val_array
    del train_dataset, val_dataset

    clear_device_cache()


STARTING SPLIT: split_01_seed_11
Loaded:
  Train: train_no_dr_scaled_v5_voltage_random10_new_features_split_01_seed_11.parquet (698993, 19)
  Val  : val_no_dr_scaled_v5_voltage_random10_new_features_split_01_seed_11.parquet (133277, 19)
  Test : test_no_dr_scaled_v5_voltage_random10_new_features_split_01_seed_11.parquet (134655, 19)
Data validation passed.
Clean normal training windows   : 24,817
Clean normal validation windows : 4,686

--------------------------------------------------------------------------------
split_01_seed_11 | 1/6 | LSTM-AE
--------------------------------------------------------------------------------
LSTMAutoencoder(
  (encoder_lstm): LSTM(16, 64, batch_first=True)
  (encoder_fc): Linear(in_features=64, out_features=32, bias=True)
  (decoder_input_fc): Linear(in_features=32, out_features=64, bias=True)
  (decoder_h0_fc): Linear(in_features=32, out_features=64, bias=True)
  (decoder_c0_fc): Linear(in_features=32, out_features=64, bias=True)
  (decoder_lstm):

/opt/anaconda3/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Epoch 001 | Train: 0.362283 | Val: 0.378697 | Gap: +0.016414
Epoch 010 | Train: 0.328711 | Val: 0.376276 | Gap: +0.047566
Epoch 020 | Train: 0.328666 | Val: 0.376243 | Gap: +0.047577
Epoch 030 | Train: 0.328643 | Val: 0.376203 | Gap: +0.047559
Epoch 040 | Train: 0.328599 | Val: 0.376240 | Gap: +0.047641
Epoch 050 | Train: 0.328655 | Val: 0.376198 | Gap: +0.047542
Early stopping at epoch 53. Best validation loss: 0.376176

Scoring training data...
Anomaly threshold: 0.832707 (95th percentile)
Scoring validation data...
Scoring test data...

Finished TCN-AE | Test AUROC=0.4812 | Test AUPRC=0.3960 | Separation=1.46x

--------------------------------------------------------------------------------
split_01_seed_11 | 3/6 | Transformer-AE
--------------------------------------------------------------------------------
TransformerAutoencoder(
  (input_projection): Linear(in_features=16, out_features=64, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x

/opt/anaconda3/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Epoch 001 | Train: 0.310804 | Val: 0.262468 | Gap: -0.048336
Epoch 010 | Train: 0.260352 | Val: 0.259228 | Gap: -0.001124
Epoch 020 | Train: 0.260276 | Val: 0.259142 | Gap: -0.001134
Epoch 030 | Train: 0.260092 | Val: 0.259130 | Gap: -0.000962
Epoch 040 | Train: 0.260191 | Val: 0.259067 | Gap: -0.001124
Epoch 050 | Train: 0.260123 | Val: 0.259053 | Gap: -0.001070
Early stopping at epoch 53. Best validation loss: 0.259052

Scoring training data...
Anomaly threshold: 0.501741 (95th percentile)
Scoring validation data...
Scoring test data...

Finished TCN-AE | Test AUROC=0.3008 | Test AUPRC=0.2158 | Separation=0.87x

--------------------------------------------------------------------------------
split_02_seed_22 | 3/6 | Transformer-AE
--------------------------------------------------------------------------------
TransformerAutoencoder(
  (input_projection): Linear(in_features=16, out_features=64, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x

...

Epoch 001 | Train: 0.359350 | Val: 0.335316 | Gap: -0.024034
Epoch 010 | Train: 0.282824 | Val: 0.318983 | Gap: +0.036158
Epoch 020 | Train: 0.282523 | Val: 0.307697 | Gap: +0.025174
Epoch 030 | Train: 0.282320 | Val: 0.305535 | Gap: +0.023215
Epoch 040 | Train: 0.282290 | Val: 0.304028 | Gap: +0.021738
Epoch 050 | Train: 0.282313 | Val: 0.304934 | Gap: +0.022621
Early stopping at epoch 55. Best validation loss: 0.304028

Scoring training data...
Anomaly threshold: 0.565723 (95th percentile)
Scoring validation data...
Scoring test data...

Finished TCN-AE | Test AUROC=0.5865 | Test AUPRC=0.5069 | Separation=4.62x

--------------------------------------------------------------------------------
split_08_seed_88 | 3/6 | Transformer-AE
--------------------------------------------------------------------------------
TransformerAutoencoder(
  (input_projection): Linear(in_features=16, out_features=64, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x

/opt/anaconda3/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Epoch 001 | Train: 0.299374 | Val: 0.254332 | Gap: -0.045042
Epoch 010 | Train: 0.245013 | Val: 0.251662 | Gap: +0.006649
Epoch 020 | Train: 0.244880 | Val: 0.251563 | Gap: +0.006683
Epoch 030 | Train: 0.244860 | Val: 0.251616 | Gap: +0.006755
Epoch 040 | Train: 0.244858 | Val: 0.251531 | Gap: +0.006672
Early stopping at epoch 48. Best validation loss: 0.251520

Scoring training data...
Anomaly threshold: 0.467401 (95th percentile)
Scoring validation data...
Scoring test data...

Finished TCN-AE | Test AUROC=0.5263 | Test AUPRC=0.3165 | Separation=1.01x

--------------------------------------------------------------------------------
split_09_seed_99 | 3/6 | Transformer-AE
--------------------------------------------------------------------------------
TransformerAutoencoder(
  (input_projection): Linear(in_features=16, out_features=64, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAtten

/opt/anaconda3/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Epoch 001 | Train: 0.312825 | Val: 0.252676 | Gap: -0.060150
Epoch 010 | Train: 0.253779 | Val: 0.248277 | Gap: -0.005502
Epoch 020 | Train: 0.254043 | Val: 0.248204 | Gap: -0.005839
Epoch 030 | Train: 0.253615 | Val: 0.248119 | Gap: -0.005496
Epoch 040 | Train: 0.251190 | Val: 0.248008 | Gap: -0.003181
Epoch 050 | Train: 0.251398 | Val: 0.248315 | Gap: -0.003084
Early stopping at epoch 51. Best validation loss: 0.247995

Scoring training data...
Anomaly threshold: 0.468812 (95th percentile)
Scoring validation data...
Scoring test data...

Finished TCN-AE | Test AUROC=0.4737 | Test AUPRC=0.3789 | Separation=1.02x

--------------------------------------------------------------------------------
split_10_seed_111 | 3/6 | Transformer-AE
--------------------------------------------------------------------------------
TransformerAutoencoder(
  (input_projection): Linear(in_features=16, out_features=64, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 

## 12. Cross-split architecture summary

In [36]:
if not ALL_METRICS:
    raise RuntimeError(
        "No model metrics were created or loaded."
    )

all_metrics_df = pd.concat(
    ALL_METRICS,
    ignore_index=True,
)

all_flight_summary_df = pd.concat(
    ALL_FLIGHT_SUMMARIES,
    ignore_index=True,
)

all_metrics_path = (
    MASTER_OUTPUT_DIR
    / "all_architectures_all_splits_metrics.csv"
)

all_flight_path = (
    MASTER_OUTPUT_DIR
    / "all_architectures_all_splits_flight_summary.csv"
)

all_metrics_df.to_csv(
    all_metrics_path,
    index=False,
)

all_flight_summary_df.to_csv(
    all_flight_path,
    index=False,
)

test_metrics = all_metrics_df[
    all_metrics_df["data_split"] == "Test"
].copy()

summary_df = (
    test_metrics
    .groupby("architecture")
    .agg(
        splits_completed=("split", "nunique"),
        flight_auroc_mean=("flight_auroc", "mean"),
        flight_auroc_std=("flight_auroc", "std"),
        flight_auprc_mean=("flight_auprc", "mean"),
        flight_auprc_std=("flight_auprc", "std"),
        separation_mean=("separation_ratio", "mean"),
        separation_std=("separation_ratio", "std"),
        best_val_loss_mean=("best_val_loss", "mean"),
        training_seconds_mean=("training_seconds", "mean"),
        n_params=("n_params", "first"),
    )
    .sort_values(
        [
            "flight_auroc_mean",
            "flight_auprc_mean",
        ],
        ascending=False,
    )
    .reset_index()
)

summary_path = (
    MASTER_OUTPUT_DIR
    / "architecture_10split_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

print("\n" + "=" * 100)
print("ALTERNATIVE AUTOENCODER — 10-SPLIT TEST SUMMARY")
print("=" * 100)

display(
    summary_df.round(4)
)

best_model = summary_df.iloc[0]

print(
    f"\nBest architecture: "
    f"{best_model['architecture']}"
)
print(
    f"Mean test AUROC : "
    f"{best_model['flight_auroc_mean']:.4f} "
    f"± {best_model['flight_auroc_std']:.4f}"
)
print(
    f"Mean test AUPRC : "
    f"{best_model['flight_auprc_mean']:.4f} "
    f"± {best_model['flight_auprc_std']:.4f}"
)

print("\nSaved:")
print(all_metrics_path)
print(all_flight_path)
print(summary_path)


ALTERNATIVE AUTOENCODER — 10-SPLIT TEST SUMMARY


,architecture,splits_completed,flight_auroc_mean,flight_auroc_std,flight_auprc_mean,flight_auprc_std,separation_mean,separation_std,best_val_loss_mean,training_seconds_mean,n_params
0,Transformer-AE,10,0.6143,0.1375,0.5544,0.1466,15939.3182,36057.9095,0.0001,428.3080,114896
1,LSTM-AE,10,0.5526,0.1201,0.4251,0.1260,12.9753,24.3908,0.0440,825.3301,63728
2,CNN-LSTM-AE,10,0.5444,0.1216,0.4115,0.1178,10.7546,18.5494,0.0586,704.9843,68272
3,TCN-AE,10,0.5278,0.1269,0.4330,0.1315,2.2289,1.7243,0.2916,167.7713,30144
4,VAE-LSTM,10,0.5278,0.1215,0.3804,0.1165,10.4997,22.0213,0.0688,738.8331,61584
5,BiLSTM-AE,10,0.5211,0.1026,0.3737,0.1108,13.1597,21.6026,0.0392,994.0756,82544



Best architecture: Transformer-AE
Mean test AUROC : 0.6143 ± 0.1375
Mean test AUPRC : 0.5544 ± 0.1466

Saved:
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/alternative_ae_10splits_v5_voltage_random10_new_features/all_architectures_all_splits_metrics.csv
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/alternative_ae_10splits_v5_voltage_random10_new_features/all_architectures_all_splits_flight_summary.csv
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/alternative_ae_10splits_v5_voltage_random10_new_features/architecture_10split_summary.csv


In [38]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

data = {
    'LSTM-AE':        {'auroc': [0.5639,0.4060,0.4962,0.4812,0.6992,0.5113,0.7143,0.6316,0.6541,0.3684],
                       'auprc': [0.4550,0.2446,0.3021,0.3880,0.6611,0.3913,0.4918,0.5812,0.3863,0.3492],
                       'sep':   [3.47,  0.32,  0.15,  1.40,  16.06, 78.49, 4.49,  24.15, 0.46,  0.75]},
    'TCN-AE':         {'auroc': [0.4812,0.3008,0.5940,0.3684,0.5639,0.6917,0.6917,0.5865,0.5263,0.4737],
                       'auprc': [0.3960,0.2158,0.4089,0.3526,0.5158,0.5887,0.6496,0.5069,0.3165,0.3789],
                       'sep':   [1.46,  0.87,  1.05,  0.91,  4.32,  5.07,  1.97,  4.62,  1.01,  1.02]},
    'Transformer-AE': {'auroc': [0.5865,0.5564,0.6466,0.4135,0.7519,0.5489,0.8045,0.8195,0.4962,0.5188],
                       'auprc': [0.6320,0.3358,0.4753,0.3955,0.6830,0.5455,0.7405,0.7690,0.5037,0.4636],
                       'sep':   [716.28,0.09,  0.27,  2.67,  170.36,108628.50,457.25,49409.24,5.74,2.77]},
    'CNN-LSTM-AE':    {'auroc': [0.5038,0.3835,0.4887,0.4737,0.6541,0.5188,0.7293,0.6692,0.6391,0.3835],
                       'auprc': [0.3904,0.2373,0.2986,0.3117,0.5497,0.3932,0.5049,0.6005,0.4753,0.3535],
                       'sep':   [1.34,  0.40,  0.23,  1.14,  12.77, 59.17, 5.68,  23.38, 2.40,  1.04]},
    'VAE-LSTM':       {'auroc': [0.5038,0.3759,0.4887,0.5263,0.6391,0.5113,0.7143,0.6090,0.6015,0.3083],
                       'auprc': [0.3904,0.2357,0.2986,0.3361,0.5454,0.3911,0.4904,0.5432,0.3538,0.2195],
                       'sep':   [2.33,  0.30,  0.18,  0.97,  7.36,  71.34, 4.06,  17.32, 0.78,  0.35]},
    'BiLSTM-AE':      {'auroc': [0.5263,0.4286,0.4737,0.4812,0.6466,0.5188,0.6466,0.5789,0.5940,0.3158],
                       'auprc': [0.3954,0.2522,0.2948,0.3185,0.5932,0.3950,0.4473,0.4670,0.3508,0.2230],
                       'sep':   [3.10,  0.15,  0.17,  1.28,  17.63, 61.92, 4.53,  41.89, 0.55,  0.39]},
}

# --- Combined score: AUPRC * log1p(separation) per split ---
# log1p prevents explosion from huge separation values
# then take mean and std across splits

results = []
for arch, vals in data.items():
    combined = [a * np.log1p(s) for a, s in zip(vals['auprc'], vals['sep'])]
    results.append({
        'architecture': arch,
        'auprc_mean':   np.mean(vals['auprc']),
        'auprc_std':    np.std(vals['auprc']),
        'auroc_mean':   np.mean(vals['auroc']),
        'sep_mean':     np.mean(vals['sep']),
        'sep_std':      np.std(vals['sep']),
        'combined_mean':np.mean(combined),
        'combined_std': np.std(combined),
        'combined_all': combined,
    })

results.sort(key=lambda x: x['combined_mean'], reverse=True)

print(f"\n{'Architecture':<18} {'AUPRC mean':>12} {'AUPRC std':>10} {'Sep mean':>10} {'Combined mean':>14} {'Combined std':>13}")
print('-' * 80)
for r in results:
    print(f"{r['architecture']:<18} {r['auprc_mean']:>12.4f} {r['auprc_std']:>10.4f} "
          f"{r['sep_mean']:>10.2f} {r['combined_mean']:>14.4f} {r['combined_std']:>13.4f}")

print(f"\n✓ Winner by AUPRC × log(separation): {results[0]['architecture']}")
print(f"  Combined score: {results[0]['combined_mean']:.4f} ± {results[0]['combined_std']:.4f}")

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2196F3','#FF9800','#9C27B0','#4CAF50','#F44336','#00BCD4']
arch_names = [r['architecture'] for r in results]

# Plot 1: AUPRC boxplot
auprc_vals = [data[a]['auprc'] for a in arch_names]
bp1 = axes[0].boxplot(auprc_vals, patch_artist=True,
                      medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for i, (vals, color) in enumerate(zip(auprc_vals, colors)):
    x = np.random.normal(i+1, 0.04, size=len(vals))
    axes[0].scatter(x, vals, alpha=0.8, color=color, edgecolors='black', s=35, zorder=3)
axes[0].set_xticks(range(1, len(arch_names)+1))
axes[0].set_xticklabels(arch_names, rotation=20, ha='right', fontsize=8)
axes[0].set_ylabel('AUPRC')
axes[0].set_title('AUPRC Distribution', fontweight='bold')
axes[0].axhline(0.5, color='black', linestyle='--', alpha=0.3)
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: log(separation) boxplot
log_sep_vals = [[np.log1p(s) for s in data[a]['sep']] for a in arch_names]
bp2 = axes[1].boxplot(log_sep_vals, patch_artist=True,
                      medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for i, (vals, color) in enumerate(zip(log_sep_vals, colors)):
    x = np.random.normal(i+1, 0.04, size=len(vals))
    axes[1].scatter(x, vals, alpha=0.8, color=color, edgecolors='black', s=35, zorder=3)
axes[1].set_xticks(range(1, len(arch_names)+1))
axes[1].set_xticklabels(arch_names, rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('log1p(Separation)')
axes[1].set_title('log(Separation) Distribution', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Combined score boxplot
combined_vals = [r['combined_all'] for r in results]
bp3 = axes[2].boxplot(combined_vals, patch_artist=True,
                      medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp3['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for i, (vals, color) in enumerate(zip(combined_vals, colors)):
    x = np.random.normal(i+1, 0.04, size=len(vals))
    axes[2].scatter(x, vals, alpha=0.8, color=color, edgecolors='black', s=35, zorder=3)
axes[2].set_xticks(range(1, len(arch_names)+1))
axes[2].set_xticklabels(arch_names, rotation=20, ha='right', fontsize=8)
axes[2].set_ylabel('AUPRC × log1p(Separation)')
axes[2].set_title('Combined Score Distribution', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Architecture Comparison — AUPRC × log(Separation) Combined Criterion',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('architecture_combined_score.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: architecture_combined_score.png')


Architecture         AUPRC mean  AUPRC std   Sep mean  Combined mean  Combined std
--------------------------------------------------------------------------------
Transformer-AE           0.5544     0.1391   15939.32         2.9076        2.7627
LSTM-AE                  0.4251     0.1195      12.97         0.7772        0.7258
CNN-LSTM-AE              0.4115     0.1117      10.76         0.7474        0.6530
BiLSTM-AE                0.3737     0.1051      13.16         0.7020        0.6955
VAE-LSTM                 0.3804     0.1106      10.50         0.6286        0.6038
TCN-AE                   0.4330     0.1247       2.23         0.5007        0.3214

✓ Winner by AUPRC × log(separation): Transformer-AE
  Combined score: 2.9076 ± 2.7627
Saved: architecture_combined_score.png


/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_64838/2689949486.py:112: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [39]:
results = []
for arch, vals in data.items():
    combined = [a * np.log1p(s) for a, s in zip(vals['auprc'], vals['sep'])]
    results.append({
        'architecture':    arch,
        'auprc_median':    np.median(vals['auprc']),
        'auprc_mean':      np.mean(vals['auprc']),
        'sep_median':      np.median(vals['sep']),
        'combined_median': np.median(combined),
        'combined_iqr':    np.percentile(combined, 75) - np.percentile(combined, 25),
        'combined_all':    combined,
    })

results.sort(key=lambda x: x['combined_median'], reverse=True)

print(f"\n{'Architecture':<18} {'AUPRC median':>13} {'Sep median':>11} {'Combined median':>16} {'IQR':>8}")
print('-' * 75)
for r in results:
    print(f"{r['architecture']:<18} {r['auprc_median']:>13.4f} "
          f"{r['sep_median']:>11.2f} {r['combined_median']:>16.4f} {r['combined_iqr']:>8.4f}")

print(f"\n✓ Winner by median AUPRC × log(separation): {results[0]['architecture']}")


Architecture        AUPRC median  Sep median  Combined median      IQR
---------------------------------------------------------------------------
Transformer-AE            0.5246       88.05           2.2371   3.9025
LSTM-AE                   0.3896        2.44           0.5105   1.3350
CNN-LSTM-AE               0.3918        1.87           0.4568   1.0800
BiLSTM-AE                 0.3729        2.19           0.4102   1.3248
VAE-LSTM                  0.3721        1.65           0.3488   0.9670
TCN-AE                    0.4024        1.25           0.3250   0.5857

✓ Winner by median AUPRC × log(separation): Transformer-AE


In [40]:


data = {
    'LSTM-AE':        {'auprc': [0.4550,0.2446,0.3021,0.3880,0.6611,0.3913,0.4918,0.5812,0.3863,0.3492],
                       'sep':   [3.47,  0.32,  0.15,  1.40,  16.06, 78.49, 4.49,  24.15, 0.46,  0.75]},
    'TCN-AE':         {'auprc': [0.3960,0.2158,0.4089,0.3526,0.5158,0.5887,0.6496,0.5069,0.3165,0.3789],
                       'sep':   [1.46,  0.87,  1.05,  0.91,  4.32,  5.07,  1.97,  4.62,  1.01,  1.02]},
    'Transformer-AE': {'auprc': [0.6320,0.3358,0.4753,0.3955,0.6830,0.5455,0.7405,0.7690,0.5037,0.4636],
                       'sep':   [716.28,0.09,  0.27,  2.67,  170.36,108628.50,457.25,49409.24,5.74,2.77]},
    'CNN-LSTM-AE':    {'auprc': [0.3904,0.2373,0.2986,0.3117,0.5497,0.3932,0.5049,0.6005,0.4753,0.3535],
                       'sep':   [1.34,  0.40,  0.23,  1.14,  12.77, 59.17, 5.68,  23.38, 2.40,  1.04]},
    'VAE-LSTM':       {'auprc': [0.3904,0.2357,0.2986,0.3361,0.5454,0.3911,0.4904,0.5432,0.3538,0.2195],
                       'sep':   [2.33,  0.30,  0.18,  0.97,  7.36,  71.34, 4.06,  17.32, 0.78,  0.35]},
    'BiLSTM-AE':      {'auprc': [0.3954,0.2522,0.2948,0.3185,0.5932,0.3950,0.4473,0.4670,0.3508,0.2230],
                       'sep':   [3.10,  0.15,  0.17,  1.28,  17.63, 61.92, 4.53,  41.89, 0.55,  0.39]},
}

print(f"\n{'Architecture':<18} {'Splits sep>1x':>14} {'Median AUPRC':>13} {'Median sep(stable)':>19} {'Verdict':>12}")
print('-' * 82)

results = []
for arch, vals in data.items():
    sep = vals['sep']
    auprc = vals['auprc']
    
    # Count splits where separation > 1 (genuine discrimination)
    good_splits = sum(1 for s in sep if s > 1.0)
    
    # Stable median: use only splits where sep is in 1x-100x range (genuine, not collapse)
    stable_idx = [i for i, s in enumerate(sep) if 1.0 < s <= 100]
    stable_sep_median = np.median([sep[i] for i in stable_idx]) if stable_idx else 0
    stable_auprc_median = np.median([auprc[i] for i in stable_idx]) if stable_idx else 0
    combined = stable_auprc_median * np.log1p(stable_sep_median) if stable_idx else 0
    
    verdict = 'UNSTABLE' if good_splits < 6 else 'OK'
    
    results.append({
        'architecture': arch,
        'good_splits': good_splits,
        'stable_auprc_median': stable_auprc_median,
        'stable_sep_median': stable_sep_median,
        'combined': combined,
        'verdict': verdict,
    })
    print(f"{arch:<18} {good_splits:>14} {stable_auprc_median:>13.4f} {stable_sep_median:>19.2f} {verdict:>12}")

# Filter to stable only, rank by combined
stable = [r for r in results if r['verdict'] == 'OK']
stable.sort(key=lambda x: x['combined'], reverse=True)

print(f"\n✓ Winner (stable architectures only): {stable[0]['architecture']}")
print(f"  Stable splits AUPRC median: {stable[0]['stable_auprc_median']:.4f}")
print(f"  Stable splits sep median:   {stable[0]['stable_sep_median']:.2f}x")


Architecture        Splits sep>1x  Median AUPRC  Median sep(stable)      Verdict
----------------------------------------------------------------------------------
LSTM-AE                         6        0.4734               10.27           OK
TCN-AE                          8        0.4579                1.71           OK
Transformer-AE                  8        0.4636                2.77           OK
CNN-LSTM-AE                     8        0.4343                4.04           OK
VAE-LSTM                        5        0.4904                7.36     UNSTABLE
BiLSTM-AE                       6        0.4214               11.08           OK

✓ Winner (stable architectures only): LSTM-AE
  Stable splits AUPRC median: 0.4734
  Stable splits sep median:   10.27x


In [2]:
from pathlib import Path
import pandas as pd

BASE_DIR   = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")
FOLDER_VER = "v3_voltage_random10_issue_flight-removed"
FILE_VER   = "v4_voltage_random10_cleaned"
DATA_DIR   = BASE_DIR / f"model_ready_no_dr_{FOLDER_VER}"

# Check across all 10 splits
rows = []
for split_name in [f"split_{str(i).zfill(2)}_seed_{s}" 
                   for i, s in enumerate([11,22,33,44,55,66,77,88,99,111], 1)]:
    test_df = pd.read_parquet(
        DATA_DIR / split_name / f"test_no_dr_scaled_{FILE_VER}_{split_name}.parquet",
        columns=["dataset_label"]
    )
    vc = test_df["dataset_label"].value_counts(normalize=True)
    rows.append({"split": split_name, 
                 "normal_%":   round(vc.get("normal",   0)*100, 1),
                 "abnormal_%": round(vc.get("abnormal", 0)*100, 1),
                 "total_rows": len(test_df)})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f"\nMean abnormal %: {summary['abnormal_%'].mean():.1f}%")

            split  normal_%  abnormal_%  total_rows
 split_01_seed_11      67.4        32.6      134655
 split_02_seed_22      63.0        37.0      136187
 split_03_seed_33      69.6        30.4      170021
 split_04_seed_44      72.8        27.2      159758
 split_05_seed_55      70.0        30.0      162502
 split_06_seed_66      61.6        38.4      141800
 split_07_seed_77      62.8        37.2      150357
 split_08_seed_88      56.8        43.2      163495
 split_09_seed_99      71.0        29.0      130512
split_10_seed_111      72.8        27.2      151918

Mean abnormal %: 33.2%


So random classifier AUPRC baseline = 0.332

In [4]:
from pathlib import Path
import torch

ckpt = torch.load(
    "/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/architecture_comparison_v3_voltage_random10_issue_flight-removed/best_lstm_ae.pt",
    map_location="cpu", weights_only=False
)
print(type(ckpt))
if isinstance(ckpt, dict):
    print(list(ckpt.keys()))
    # check input size from first layer
    for k,v in ckpt.items():
        if "weight" in k:
            print(k, v.shape)
            break

<class 'collections.OrderedDict'>
['encoder_lstm.weight_ih_l0', 'encoder_lstm.weight_hh_l0', 'encoder_lstm.bias_ih_l0', 'encoder_lstm.bias_hh_l0', 'encoder_fc.weight', 'encoder_fc.bias', 'decoder_input_fc.weight', 'decoder_input_fc.bias', 'decoder_h0_fc.weight', 'decoder_h0_fc.bias', 'decoder_c0_fc.weight', 'decoder_c0_fc.bias', 'decoder_lstm.weight_ih_l0', 'decoder_lstm.weight_hh_l0', 'decoder_lstm.bias_ih_l0', 'decoder_lstm.bias_hh_l0', 'output_layer.weight', 'output_layer.bias']
encoder_lstm.weight_ih_l0 torch.Size([256, 13])
